# Acoustic Signal Analytics for Welding Process Monitoring\n\nThis notebook demonstrates the portfolio pipeline: audio discovery → feature extraction → anomaly detection → visual analysis.

In [ ]:
from pathlib import Path\nimport sys\n\nROOT = Path.cwd().resolve()\nif ROOT.name == 'notebooks':\n    ROOT = ROOT.parent\nsys.path.insert(0, str(ROOT))\n\nfrom src.features import discover_audio_files, build_feature_table\nfrom src.anomaly import score_anomalies

In [ ]:
data_root = ROOT / 'welding-data'\naudio_files = discover_audio_files(data_root)\nprint(f'Audio files found: {len(audio_files)}')\naudio_files[:10]

## Feature Engineering\n\nEach recording is segmented into 2-second windows with a 1-second hop. Source identity and timestamps are retained so future supervised experiments can split by recording.

In [ ]:
features = build_feature_table(\n    audio_files,\n    sample_rate=22_050,\n    window_sec=2.0,\n    hop_sec=1.0,\n)\nprint(features.shape)\nfeatures.head()

## Unsupervised Anomaly Detection\n\nIsolation Forest produces a relative anomaly score for each audio window. Higher scores indicate more unusual feature combinations within the analysed population.

In [ ]:
scores, model = score_anomalies(features)\nscores[['file_name', 'start_sec', 'anomaly_score', 'is_anomaly']].head(20)

In [ ]:
import matplotlib.pyplot as plt\n\ntop = scores.head(20).sort_values('anomaly_score')\nplt.figure(figsize=(10, 7))\nplt.barh(top['file_name'] + ' @ ' + top['start_sec'].round(1).astype(str) + 's', top['anomaly_score'])\nplt.xlabel('Relative anomaly score')\nplt.ylabel('Window')\nplt.title('Most unusual acoustic windows')\nplt.tight_layout()\nplt.show()

## Interpretation\n\nTreat the anomaly score as a prioritisation signal, not as a failure probability. A production deployment would require controlled experiments, documented process labels, domain validation and an operating threshold.